```SQL
SELECT d.name,f.name AS "Floor",e.name AS "Exhibit", count(*) AS "Interactions", ROUND(AVG(b.value),2) AS "Average Rating"
FROM kiosk_output AS ko
JOIN exhibit e USING(exhibit_id)
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
JOIN floor f USING(floor_id)
JOIN department d USING(department_id)
WHERE bt.name = 'rating'
GROUP BY f.name,e.name, d.name
ORDER BY  "Average Rating" DESC
```

# Museum Analysis

This notebook connects to the museum database and conducts simple analysis of the data.

## Imports

In [ ]:
from os import environ, _Environ

import pandas as pd
from dotenv import load_dotenv
from psycopg2 import connect
import altair as alt
from sqlalchemy import create_engine

## Setup

In [ ]:

engine = create_engine(
    f"postgresql://{environ.get('DATABASE_USER')}:{environ.get('DATABASE_PASSWORD')}@{environ.get('DATABASE_HOST')}:{environ.get('DATABASE_PORT')}/{environ.get('DATABASE_DB')}"
)   

## Querying data

### Data questions to answer for Angela:
- Which exhibits are the most popular? (number of visitors and average rating) 
- Which exhibit types are the most popular? (number of visitors per exhibit and average rating) 
<!-- ###
- How have different exhibits performed over time? (number of visitors and average rating)
- How have different exhibit types performed over time? (number of visitors and average rating) -->
###
- How does the time of day effect visitor satisfaction?

### Data questions to answer for Rita:
- Which exhibitions generate the most assistance button presses
- Which exhibitions generate the most emergency button presses
###
- How does assistance and emergency occurrence vary throughout the day

In [ ]:
# Which exhibits are the most popular? (interactions)
# query to get the number of button presses for each exhibit
query = """
SELECT e.name, COUNT(*)
FROM kiosk_output ko
JOIN exhibit e USING(exhibit_id)
GROUP BY e.name
ORDER BY COUNT(*) DESC
;
"""
# execute the query and load the results into a pandas dataframe

df = pd.read_sql_query(query, engine)

# create a bar chart of the results
chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('count', title='Number of Kiosk Interactions'),
    y=alt.Y('name', title='Exhibit', sort='-x'),
    color=alt.Color('name:N', legend=None),
).properties(
    title='Number of Kiosk Interactions for Each Exhibit',
    
)

chart.save('visualisations/number_of_kiosk_interactions_for_each_exhibit.png')
chart

In [ ]:
# Which exhibits are the most popular? (rating)
# query to get the number of button presses for each exhibit
query = """
SELECT e.name, ROUND(AVG(b.value),2) as average_rating
FROM kiosk_output ko
JOIN exhibit e USING(exhibit_id)
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
WHERE bt.name = 'rating'
GROUP BY e.name
ORDER BY average_rating DESC
;
"""
# execute the query and load the results into a pandas dataframe
df = pd.read_sql_query(query, engine)

# create a bar chart of the results
chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('average_rating:Q', title='Average Rating'),
    y=alt.Y('name:N', title='Exhibit', sort='-x'),
    color=alt.Color('name:N', legend=None),
).properties(
    title='Average Rating for Each Exhibit',
    
)
# df.desc
chart.save('visualisations/average_rating_for_each_exhibit.png')
chart.show()

In [ ]:
# Which exhibit types are the most popular? (interactions)
# query to get the number of button presses for each exhibit type
query = """
-- get the proportion of interactions for each exhibit type
-- we use a window function to get the total number of exhibits and then divide the number of interactions for each exhibit type by the number of exhibits of that type
SELECT d.name, COUNT(*)/count(DISTINCT(e.name)) as average_interactions
FROM kiosk_output ko
JOIN exhibit e USING(exhibit_id)
JOIN department d USING(department_id)
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
WHERE bt.name = 'rating'
GROUP BY d.name
ORDER BY average_interactions DESC
;
"""
# execute the query and load the results into a pandas dataframe
df = pd.read_sql_query(query, engine)

# create a bar chart of the results
chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('average_interactions:Q', title='Average Interactions per Exhibit'),
    y=alt.Y('name:N', title='Exhibit Type', sort='-x'),
    color=alt.Color('name:N', legend=None),
).properties(
    title='Average Number of Interactions per Exhibit for Each Exhibit Type',

)
# df.desc
chart.save('visualisations/average_number_of_interactions_per_exhibit_for_each_exhibit_type.png')
chart.show()

In [ ]:
# Which exhibit types are the most popular? (rating)
# query to get the number of button presses for each exhibit type
query = """
SELECT d.name, ROUND(AVG(b.value),2) as average_rating
FROM kiosk_output ko
JOIN exhibit e USING(exhibit_id)
JOIN department d USING(department_id)
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
WHERE bt.name = 'rating'
GROUP BY d.name
ORDER BY average_rating DESC
;
"""
# execute the query and load the results into a pandas dataframe
df = pd.read_sql_query(query, engine)

# create a bar chart of the results
chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('average_rating:Q', title='Average Rating'),
    y=alt.Y('name:N', title='Exhibit Type', sort='-x'),
    color=alt.Color('name:N', legend=None),
).properties(
    title='Average Rating for Each Exhibit Type',

)
# df.desc
chart.save('visualisations/average_rating_for_each_exhibit_type.png')
chart.show()

In [ ]:
# How does the time of day effect visitor satisfaction?

query = """
SELECT DATE_PART('hour', ko.recorded_at) as hour, ROUND(AVG(b.value+1),2) as average_rating
FROM kiosk_output ko
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
WHERE bt.name = 'rating'
GROUP BY HOUR
ORDER BY HOUR
;
"""

df = pd.read_sql_query(query, engine)

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('hour:O', title='Hour of Day'),
    y=alt.Y('average_rating:Q', title='Average Rating'),
    color=alt.Color('hour:N', legend=None),
).properties(
    title='Average Rating by Hour of Day',
    width=400,
    height=200
)
df
chart.save('visualisations/average_rating_by_hour_of_day.png')
chart

In [ ]:
# How does the time of day effect visitor satisfaction?

query = """
SELECT DATE_PART('hour', ko.recorded_at) as hour, ROUND(AVG(b.value+1),2) as average_rating
FROM kiosk_output ko
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
WHERE bt.name = 'rating'
GROUP BY HOUR
ORDER BY HOUR
;
"""

df = pd.read_sql_query(query, engine)

chart = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X('hour:O', title='Hour of Day'),
    y=alt.Y('average_rating:Q', title='Average Rating', scale=alt.Scale(zero=False, domainMin=2.5, domainMax=3.5)),
).properties(
    title='Average Rating by Hour of Day',
)
# save the chart as a PNG file
chart.save('visualisations/average_rating_by_hour_line_chart.png')
chart

In [ ]:
# # How does the time of day effect visitor satisfaction?

# # time of all ratings scatter plot
# query = """
# SELECT ko.recorded_at::time as time, b.value+1 as rating
# FROM kiosk_output ko
# JOIN button b USING(button_id)
# JOIN button_type bt USING(button_type_id)
# WHERE bt.name = 'rating'
# ;
# """

# df = pd.read_sql_query(query, engine)

# chart = alt.Chart(df).mark_circle(size=60).encode(
#     x=alt.X('time:T', title='Time of Day'),
#     y=alt.Y('rating:Q', title='Rating', scale=alt.Scale(zero=False, domainMin=1.5, domainMax=3.5)),
#     color=alt.Color('rating:Q', scale=alt.Scale(scheme='viridis'), legend=None),
# ).properties(
#     title='Ratings by Time of Day',
#     width=400,
#     height=200
# )   
# chart.show()

In [ ]:
# Which exhibitions generate the most assistance button presses
query = """
SELECT e.name, COUNT(*)
FROM kiosk_output ko
JOIN exhibit e USING(exhibit_id)
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
WHERE bt.name = 'other' AND b.description = 'assistance'
GROUP BY e.name
ORDER BY COUNT(*) DESC
;
"""
df = pd.read_sql_query(query, engine)

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('count', title='Number of Assistance Button Presses'),
    y=alt.Y('name', title='Exhibit', sort='-x'),
    color=alt.Color('name:N', legend=None),
).properties(
    title='Number of Assistance Button Presses for Each Exhibit',
)
chart.save('visualisations/number_of_assistance_button_presses_for_each_exhibit.png')
chart

In [ ]:

# Which exhibitions generate the most emergency button presses
query = """
SELECT e.name, COUNT(*)
FROM kiosk_output ko
JOIN exhibit e USING(exhibit_id)
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
WHERE bt.name = 'other' AND b.description = 'emergency'
GROUP BY e.name
ORDER BY COUNT(*) DESC
;
"""
df = pd.read_sql_query(query, engine)

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('count', title='Number of Emergency Button Presses'),
    y=alt.Y('name', title='Exhibit', sort='-x'),
    color=alt.Color('name:N', legend=None),
).properties(
    title='Number of Emergency Button Presses for Each Exhibit',
)
chart.save(
    'visualisations/number_of_assistance_button_presses_for_each_exhibit.png')
chart

There were only 2 instances where a visitor pressed the emergency button but they were both at 

In [ ]:
# How does assistance and emergency occurrence vary throughout the day
query = """
SELECT DATE_PART('hour', ko.recorded_at) as hour,
COUNT(*) as assistance_count
FROM kiosk_output ko
JOIN button b USING(button_id)
JOIN button_type bt USING(button_type_id)
WHERE b.description = 'assistance'
GROUP BY hour
ORDER BY hour
;
"""
df = pd.read_sql_query(query, engine)

chart = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X('hour:O', title='Hour of Day (24hr)'),
    y=alt.Y('assistance_count:Q', title='Number of Assistance Button Presses'),
    color=alt.value('blue'),
).properties(
    title='Number of Assistance Button Presses by Hour of Day',
)
chart.save('visualisations/number_of_assistance_button_presses_by_hour_of_day.png')
chart

## Close Connection to Database

In [ ]:
engine.dispose()